In [ ]:
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from sklearn.metrics import classification_report

In [2]:
print("Loading dataset...")
ds = load_dataset("AmazonScience/massive", "en-US")
num_labels = ds["train"].features["intent"].num_classes
print(f"Loaded. Intents: {num_labels}, Train size: {len(ds['train'])}")

Loading dataset...


Generating test split: 100%|██████████| 2974/2974 [00:00<00:00, 315346.85 examples/s]

Loaded. Intents: 60, Train size: 11514


In [3]:
print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained("bert-base-uncased")

Loading tokenizer...


In [4]:
def make_dataset(split, shuffle=False):
    enc = tok(
        ds[split]["utt"],
        truncation=True,
        padding=True,
        max_length=32,
        return_tensors="tf"
    )
    dataset = tf.data.Dataset.from_tensor_slices((dict(enc), ds[split]["intent"]))
    if shuffle:
        dataset = dataset.shuffle(1000)
    return dataset.batch(32)

print("Preparing data...")
train_ds = make_dataset("train", shuffle=True)
val_ds   = make_dataset("validation")
test_ds  = make_dataset("test")

Preparing data...


2026-06-03 20:04:23.456752: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-06-03 20:04:23.456848: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-06-03 20:04:23.456861: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-06-03 20:04:23.457269: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-03 20:04:23.457536: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
print("Loading BERT...")
model = TFAutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

Loading BERT...


All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [7]:
print("Training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

Training...
Epoch 1/3


2026-06-03 20:20:56.533555: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


360/360 [==============================] - ETA: 0s - loss: 2.8472 - accuracy: 0.3521

2026-06-03 20:23:56.944679: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


360/360 [==============================] - 208s 548ms/step - loss: 2.8472 - accuracy: 0.3521 - val_loss: 2.4651 - val_accuracy: 0.4737
Epoch 2/3
360/360 [==============================] - 212s 588ms/step - loss: 1.5452 - accuracy: 0.6778 - val_loss: 1.5876 - val_accuracy: 0.6601
Epoch 3/3
360/360 [==============================] - 228s 633ms/step - loss: 0.9174 - accuracy: 0.8239 - val_loss: 1.0888 - val_accuracy: 0.7619


In [8]:
print("Evaluating...")
logits = model.predict(test_ds).logits
preds  = np.argmax(logits, axis=1)
print(classification_report(ds["test"]["intent"], preds, digits=3))

Evaluating...


2026-06-03 20:52:53.346311: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


93/93 [==============================] - 34s 337ms/step
              precision    recall  f1-score   support

           0      0.908     0.784     0.841        88
           1      0.660     0.861     0.747        36
           2      0.971     0.971     0.971        35
           3      0.000     0.000     0.000        35
           4      0.833     0.962     0.893        26
           5      0.000     0.000     0.000         1
           6      0.594     0.884     0.710        43
           7      0.000     0.000     0.000         4
           8      0.684     0.722     0.703        18
           9      0.713     0.931     0.807        72
          10      0.867     1.000     0.929        39
          11      0.789     1.000     0.882        15
          12      0.550     0.550     0.550       169
          13      0.966     0.923     0.944       156
          14      0.286     0.462     0.353        13
          15      0.000     0.000     0.000        12
          16      1.000  

/Users/keertinayak30/.conda/envs/here-nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keertinayak30/.conda/envs/here-nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/keertinayak30/.conda/envs/here-nlp/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [9]:
model.save_pretrained("intent_model")
tok.save_pretrained("intent_model")
print("Saved to intent_model/")

Saved to intent_model/


In [ ]:
from datasets import load_dataset

ds = load_dataset("AmazonScience/massive", "en-US")
intent_feature = ds["train"].features["intent"]

for i in range(10):
    utt = ds["train"][i]["utt"]
    intent_id = ds["train"][i]["intent"]
    intent_name = intent_feature.int2str(intent_id)
    print(f"{i+1}. {utt}")
    print(f"   -> {intent_name}")
    print()

1. wake me up at nine am on friday
   -> alarm_set

2. set an alarm for two hours from now
   -> alarm_set

3. olly quiet
   -> audio_volume_mute

4. stop
   -> audio_volume_mute

5. olly pause for ten seconds
   -> audio_volume_mute

6. pause for ten seconds
   -> audio_volume_mute

7. make the lighting bit more warm here
   -> iot_hue_lightchange

8. please set the lighting suitable for reading
   -> iot_hue_lightchange

9. time to sleep
   -> iot_hue_lightoff

10. time to sleep olly
   -> iot_hue_lightoff

